# 02 — Detection Experiments

This notebook compares individual hallucination detection methods:
1. NLI-based detector (DeBERTa v3)
2. Embedding similarity detector (MPNet)
3. LLM-as-judge detector (Groq/LLaMA 3)
4. Weighted ensemble

Metrics: AUROC, F1, Precision, Recall

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.utils.data_loader import load_halueval
from src.evaluation.metrics import hallucination_detection_auroc, hallucination_detection_f1

In [ ]:
# Load evaluation data
data = load_halueval(split='test')[:200]  # Subset for speed
print(f'Evaluation samples: {len(data)}')

## 1. NLI Detector

In [ ]:
from src.detection.nli_detector import NLIDetector

nli = NLIDetector()
nli_scores = []
y_true = []

for sample in tqdm(data, desc='NLI scoring'):
    score = nli.score(sample['response'], sample['context'])
    nli_scores.append(score)
    y_true.append(sample['label'])

print(f'NLI AUROC: {hallucination_detection_auroc(y_true, nli_scores)}')
print(f'NLI F1: {hallucination_detection_f1(y_true, nli_scores)}')

## 2. Embedding Detector

In [ ]:
from src.detection.embedding_detector import EmbeddingDetector

emb = EmbeddingDetector()
emb_scores = []

for sample in tqdm(data, desc='Embedding scoring'):
    score = emb.score(sample['response'], sample['context'])
    emb_scores.append(score)

print(f'Embedding AUROC: {hallucination_detection_auroc(y_true, emb_scores)}')
print(f'Embedding F1: {hallucination_detection_f1(y_true, emb_scores)}')

## 3. LLM Judge

In [ ]:
from src.detection.llm_judge import LLMJudge

judge = LLMJudge()
judge_scores = []

for sample in tqdm(data[:50], desc='LLM Judge scoring'):  # Smaller subset for API costs
    result = judge.score(sample['query'], sample['response'], sample['context'])
    judge_scores.append(result['score'])

print(f'LLM Judge AUROC: {hallucination_detection_auroc(y_true[:50], judge_scores)}')
print(f'LLM Judge F1: {hallucination_detection_f1(y_true[:50], judge_scores)}')

## 4. Ensemble Comparison

In [ ]:
# Compare all methods in a results table
results = pd.DataFrame([
    {'Method': 'NLI', 'AUROC': hallucination_detection_auroc(y_true, nli_scores),
     **hallucination_detection_f1(y_true, nli_scores)},
    {'Method': 'Embedding', 'AUROC': hallucination_detection_auroc(y_true, emb_scores),
     **hallucination_detection_f1(y_true, emb_scores)},
])
print(results.to_string(index=False))